In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

#### Lendo a tabela da camada Silver

In [0]:
tb_queimadas = spark.table('proj_queimadas.silver.silver_queimadas')

#### Criação da coluna `classe_risco_fogo`, onde fazemos a classificação de acordo com as faixas oficiais do INPE

In [0]:
tb_queimadas = (
    tb_queimadas
    .withColumns(
        {
            'classe_risco_fogo':
                    when(col('risco_fogo') <= 0.15,'Mínimo')
                    .when(col('risco_fogo') <= 0.4,'Baixo')
                    .when(col('risco_fogo') <= 0.7,'Médio')
                    .when(col('risco_fogo') <= 0.95,'Alto')
                    .otherwise('Crítico')
        }
    )
)

In [0]:
(
    tb_queimadas.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable('proj_queimadas.gold.gold_queimadas')
)

In [0]:
%sql
select * from proj_queimadas.gold.gold_queimadas